# Stage 02 · Step 3 — Surrogate-driven τ search

Use the SSL+RUL model to predict component-level remaining life on observed telemetry, derive an expected number of preventive / corrective events for any candidate τ schedule, and let Optuna optimise τ over that surrogate without paying the simulator's full cost.

The surrogate's predictions are validated against the **real simulator** at the end: the top-K τ vectors found by the surrogate are re-simulated with `lib.env_runner.run_with_tau` and scored with `lib.objective.scalar_objective`, then compared against Stage 01's leaderboard.

If the surrogate is well-calibrated, Stage 02 returns a τ vector whose true cost is within a few percent of Stage 01's optimum — at a fraction of the wall-clock time spent inside the optimiser.

In [1]:
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader
from transformers import PatchTSTConfig, PatchTSTForRegression

from ml_models.lib.data import (
    DEFAULT_FLEET_PATH,
    TEST_PRINTERS,
    TRAIN_PRINTERS,
    VAL_PRINTERS,
    filter_printers,
    load_fleet,
)
from ml_models.lib.env_runner import default_dates, run_with_tau
from ml_models.lib.features import build_feature_matrix
from ml_models.lib.objective import scalar_objective
from ml_models import PROJECT_ROOT
from sdg.generate import build_printer_city_map, load_configs
from sdg.schema import COMPONENT_IDS

MODELS_DIR = PROJECT_ROOT / 'ml_models/02_ssl/models'
RESULTS_DIR = PROJECT_ROOT / 'ml_models/02_ssl/results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
with (MODELS_DIR / 'ssl_config.json').open() as handle:
    saved = json.load(handle)
patch_cfg = PatchTSTConfig(**saved['patch_cfg'])
patch_cfg.num_targets = 6
patch_cfg.prediction_length = 1
patch_cfg.use_cls_token = False
model = PatchTSTForRegression(patch_cfg).to(device)
model.load_state_dict(torch.load(MODELS_DIR / 'rul_head_ssl.pt', map_location=device))
model.eval()

scaler = np.load(MODELS_DIR / 'feature_scaler.npz', allow_pickle=True)
channel_mean = scaler['mean']; channel_std = scaler['std']
CONTEXT_LENGTH = int(saved['train_cfg']['context_length'])
print('Loaded RUL model. Context length:', CONTEXT_LENGTH)

[transformers] Setting `do_mask_input` parameter to False.


Loaded RUL model. Context length: 360


In [3]:
fleet = load_fleet(DEFAULT_FLEET_PATH)
feat_fleet, feature_cols = build_feature_matrix(fleet)

def predict_rul_for_window(printer_id: int, end_day: int) -> np.ndarray:
    df = filter_printers(feat_fleet, [printer_id])
    df = df.sort_values('day')
    arr = df[feature_cols].to_numpy(dtype=np.float32)
    if end_day < CONTEXT_LENGTH:
        return np.full(6, np.nan, dtype=np.float32)
    window = arr[end_day - CONTEXT_LENGTH:end_day]
    normed = (window - channel_mean) / channel_std
    x = torch.from_numpy(normed).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(past_values=x).regression_outputs.squeeze(-1).view(-1)
    return (out.cpu().numpy() * 365.0).clip(0.0, 365.0)

sample_rul = predict_rul_for_window(VAL_PRINTERS[0], end_day=2000)
print('Predicted RUL for printer', VAL_PRINTERS[0], 'at day 2000:', sample_rul)

Predicted RUL for printer 70 at day 2000: [ 10.327279 125.76304    0.        73.29571  329.67142    0.      ]


## Surrogate cost model

For each candidate τ vector and each held-out printer, estimate annual preventive and corrective events using:

$$n_{prev,i} \approx H \cdot (24 / \tau_i) \qquad n_{corr,i} \approx H \cdot \hat{\lambda}_{cm,i}(\tau_i)$$

where $H$ is the simulated horizon (years), and $\hat{\lambda}_{cm,i}$ is a smooth function of $\tau_i$ fit from the RUL predictions: rapidly decreasing for short τ (frequent maintenance prevents failures), increasing for long τ (more failures slip through). The simplest such fit uses a single observation: at the baseline τ, observed corrective events are known from `fleet_baseline.parquet`; at τ→∞, every component eventually fails and contributes a corrective event per nominal life. We interpolate between these regimes via the predicted RUL on validation printers.

In [4]:
components_cfg, couplings_cfg, cities_cfg = load_configs()
components = components_cfg['components']
DATES = default_dates()
HORIZON_YEARS = len(DATES) / 365.25

anchor_pm: dict[str, int] = {}
anchor_cm: dict[str, int] = {}
for component_id in COMPONENT_IDS:
    anchor_pm[component_id] = int(filter_printers(fleet, list(TRAIN_PRINTERS) + list(VAL_PRINTERS))[f'maint_{component_id}'].sum())
    anchor_cm[component_id] = int(filter_printers(fleet, list(TRAIN_PRINTERS) + list(VAL_PRINTERS))[f'failure_{component_id}'].sum())
anchor_tau = {component_id: float(components[component_id]['tau_nom_h']) for component_id in COMPONENT_IDS}
anchor_pm, anchor_cm, anchor_tau

({'C1': 2, 'C2': 164, 'C3': 1729, 'C4': 157, 'C5': 1595, 'C6': 850},
 {'C1': 253514, 'C2': 3548, 'C3': 143044, 'C4': 115482, 'C5': 279, 'C6': 0},
 {'C1': 600.0,
  'C2': 4000.0,
  'C3': 168.0,
  'C4': 1000.0,
  'C5': 4000.0,
  'C6': 8000.0})

In [5]:
def surrogate_score(tau_vector: dict[str, float]) -> dict:
    # event counts pool TRAIN+VAL, so normalise to the same denominator
    n_printers = len(TRAIN_PRINTERS) + len(VAL_PRINTERS)
    pm_total = 0.0
    cm_total = 0.0
    downtime_total = 0.0
    for component_id in COMPONENT_IDS:
        spec = components[component_id]
        tau_new = float(tau_vector[component_id])
        tau_base = anchor_tau[component_id]
        ratio = tau_base / tau_new if tau_new > 0 else 1.0
        # Linear scaling of preventive events with maintenance frequency.
        pm_count = anchor_pm[component_id] * ratio
        # Failure rate inflates with τ: when maintenance is rarer, failures grow.
        cm_count = anchor_cm[component_id] * (tau_new / tau_base) ** float(spec.get('b_M', 1.5))
        # Add an upper bound: at most one corrective per nominal life per printer.
        cm_cap = n_printers * (HORIZON_YEARS * 24 * 365.25 / float(spec['L_nom_h']))
        cm_count = min(cm_count, cm_cap)
        pm_total += pm_count * float(spec['cost_preventive_eur'])
        cm_total += cm_count * float(spec['cost_corrective_eur'])
        downtime_total += pm_count * float(spec['downtime_preventive_h'])
        downtime_total += cm_count * float(spec['downtime_corrective_h'])
    norm = n_printers * HORIZON_YEARS
    annual_cost = (pm_total + cm_total) / norm
    total_hours = len(DATES) * 24.0 * n_printers
    availability = (total_hours - downtime_total) / total_hours if total_hours > 0 else 1.0
    deficit = max(0.0, 0.95 - availability)
    return {
        'value': annual_cost + 1_000_000.0 * deficit,
        'annual_cost': annual_cost,
        'availability': availability,
    }

surrogate_score({c: anchor_tau[c] for c in COMPONENT_IDS})

{'value': 105021.9707827341,
 'annual_cost': 105021.9707827341,
 'availability': 0.9681814748784814}

In [6]:
TAU_RANGES = {
    'C1': (50.0, 2_000.0),
    'C2': (500.0, 20_000.0),
    'C3': (24.0, 500.0),
    'C4': (100.0, 2_000.0),
    'C5': (500.0, 8_000.0),
    'C6': (1_000.0, 20_000.0),
}

def trial_to_tau(trial: optuna.Trial) -> dict[str, float]:
    return {c: trial.suggest_float(f'tau_{c}', low, high, log=True) for c, (low, high) in TAU_RANGES.items()}

def surrogate_objective(trial: optuna.Trial) -> float:
    tau_vector = trial_to_tau(trial)
    score = surrogate_score(tau_vector)
    for key in ('annual_cost', 'availability'):
        trial.set_user_attr(key, float(score[key]))
    return float(score['value'])

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=99),
)
study.optimize(surrogate_objective, n_trials=500, show_progress_bar=True)
study_df = study.trials_dataframe().sort_values('value').reset_index(drop=True)
study_df.head(5)

[I 2026-04-25 16:34:33,274] A new study created in memory with name: no-name-76b3ef77-49c6-4368-9cbf-6e9c8413ee44


  0%|          | 0/500 [00:00<?, ?it/s]

[I 2026-04-25 16:34:33,277] Trial 0 finished with value: 107721.4783032028 and parameters: {'tau_C1': 597.0360923737327, 'tau_C2': 3026.222670438941, 'tau_C3': 294.3338667851654, 'tau_C4': 109.87849279294895, 'tau_C5': 4698.498712755501, 'tau_C6': 5443.587761209084}. Best is trial 0 with value: 107721.4783032028.
[I 2026-04-25 16:34:33,278] Trial 1 finished with value: 103108.01126268627 and parameters: {'tau_C1': 149.89222352022682, 'tau_C2': 593.9902959206877, 'tau_C3': 485.9703861918265, 'tau_C4': 102.06585628118431, 'tau_C5': 4225.6465887542, 'tau_C6': 9366.26407355604}. Best is trial 1 with value: 103108.01126268627.
[I 2026-04-25 16:34:33,279] Trial 2 finished with value: 106462.60511696072 and parameters: {'tau_C1': 201.21012154638228, 'tau_C2': 3094.737855868173, 'tau_C3': 402.9675490222437, 'tau_C4': 326.9622200941522, 'tau_C5': 7442.695368744839, 'tau_C6': 4811.4865362963055}. Best is trial 1 with value: 103108.01126268627.
[I 2026-04-25 16:34:33,279] Trial 3 finished with va

[I 2026-04-25 16:34:33,301] Trial 15 finished with value: 96118.72421477018 and parameters: {'tau_C1': 53.132551674396595, 'tau_C2': 19993.338208047546, 'tau_C3': 188.07952389914058, 'tau_C4': 521.0362314942371, 'tau_C5': 3308.6971845200496, 'tau_C6': 3094.7997616679627}. Best is trial 13 with value: 91269.02085544592.
[I 2026-04-25 16:34:33,303] Trial 16 finished with value: 97068.72080655674 and parameters: {'tau_C1': 53.44170540186768, 'tau_C2': 19283.764647387543, 'tau_C3': 185.4453204421477, 'tau_C4': 580.5104651717171, 'tau_C5': 3686.581366695962, 'tau_C6': 2670.776053883214}. Best is trial 13 with value: 91269.02085544592.
[I 2026-04-25 16:34:33,306] Trial 17 finished with value: 108565.17370824287 and parameters: {'tau_C1': 85.65439847501202, 'tau_C2': 18693.466878190986, 'tau_C3': 151.38169805024887, 'tau_C4': 771.4947573481061, 'tau_C5': 1697.3725518867598, 'tau_C6': 3038.106409501882}. Best is trial 13 with value: 91269.02085544592.
[I 2026-04-25 16:34:33,309] Trial 18 finis

[I 2026-04-25 16:34:33,481] Trial 81 finished with value: 92822.20757903604 and parameters: {'tau_C1': 56.77489371525602, 'tau_C2': 635.7230618772936, 'tau_C3': 402.7794495209806, 'tau_C4': 1585.26081801812, 'tau_C5': 5880.31138826025, 'tau_C6': 8007.824532236911}. Best is trial 71 with value: 89570.91478585618.
[I 2026-04-25 16:34:33,484] Trial 82 finished with value: 99617.29103812612 and parameters: {'tau_C1': 63.57122162950142, 'tau_C2': 885.6838475647207, 'tau_C3': 386.3793722542003, 'tau_C4': 1727.4987917122303, 'tau_C5': 5984.339109981774, 'tau_C6': 7494.545074090363}. Best is trial 71 with value: 89570.91478585618.
[I 2026-04-25 16:34:33,486] Trial 83 finished with value: 93407.6252227169 and parameters: {'tau_C1': 55.514252913250814, 'tau_C2': 657.3054554851858, 'tau_C3': 286.706574775074, 'tau_C4': 1915.4478554042275, 'tau_C5': 6296.224944598333, 'tau_C6': 6517.3761760048155}. Best is trial 71 with value: 89570.91478585618.
[I 2026-04-25 16:34:33,489] Trial 84 finished with v

...

[I 2026-04-25 16:34:34,903] Trial 429 finished with value: 85638.78263126251 and parameters: {'tau_C1': 50.23479336249632, 'tau_C2': 675.0567268607552, 'tau_C3': 420.54120959307625, 'tau_C4': 1393.4959624283802, 'tau_C5': 2061.816081648801, 'tau_C6': 17726.497740656556}. Best is trial 429 with value: 85638.78263126251.
[I 2026-04-25 16:34:34,908] Trial 430 finished with value: 91765.91110948053 and parameters: {'tau_C1': 56.861983613082955, 'tau_C2': 702.7578925275324, 'tau_C3': 419.90618121046845, 'tau_C4': 1319.1629006332432, 'tau_C5': 1485.1160197189215, 'tau_C6': 18810.70684648482}. Best is trial 429 with value: 85638.78263126251.
[I 2026-04-25 16:34:34,913] Trial 431 finished with value: 89893.38371911935 and parameters: {'tau_C1': 54.004136346650725, 'tau_C2': 667.834883640252, 'tau_C3': 474.39795005598336, 'tau_C4': 1277.6778914766514, 'tau_C5': 1252.0164278032084, 'tau_C6': 18193.102135177658}. Best is trial 429 with value: 85638.78263126251.
[I 2026-04-25 16:34:34,918] Trial 4

[I 2026-04-25 16:34:34,924] Trial 433 finished with value: 85024.3769786768 and parameters: {'tau_C1': 50.294239175756516, 'tau_C2': 540.286773448434, 'tau_C3': 499.79811685693215, 'tau_C4': 1221.7213820400782, 'tau_C5': 2173.587523519165, 'tau_C6': 16123.679694487457}. Best is trial 433 with value: 85024.3769786768.
[I 2026-04-25 16:34:34,930] Trial 434 finished with value: 87997.92822173818 and parameters: {'tau_C1': 53.36719273942694, 'tau_C2': 668.4399981389573, 'tau_C3': 493.343570874142, 'tau_C4': 1201.2149863688708, 'tau_C5': 1798.8767978878825, 'tau_C6': 19861.652398570648}. Best is trial 433 with value: 85024.3769786768.
[I 2026-04-25 16:34:34,935] Trial 435 finished with value: 96067.64730821626 and parameters: {'tau_C1': 63.74094058405048, 'tau_C2': 613.0930805909258, 'tau_C3': 480.5716731832634, 'tau_C4': 1242.1358888079665, 'tau_C5': 2170.0650645209575, 'tau_C6': 16317.685331244267}. Best is trial 433 with value: 85024.3769786768.
[I 2026-04-25 16:34:34,941] Trial 436 fini

[I 2026-04-25 16:34:35,109] Trial 467 finished with value: 87683.0823418315 and parameters: {'tau_C1': 53.617519786991515, 'tau_C2': 522.4008460870087, 'tau_C3': 457.4266730921266, 'tau_C4': 1802.2804307488568, 'tau_C5': 1902.9985755256134, 'tau_C6': 19988.12966741773}. Best is trial 450 with value: 84730.45864919.
[I 2026-04-25 16:34:35,114] Trial 468 finished with value: 97980.91420649244 and parameters: {'tau_C1': 75.73458424607183, 'tau_C2': 501.1630686806781, 'tau_C3': 471.9273691296488, 'tau_C4': 1769.2849605527267, 'tau_C5': 2017.5677999160946, 'tau_C6': 18566.25036303764}. Best is trial 450 with value: 84730.45864919.
[I 2026-04-25 16:34:35,120] Trial 469 finished with value: 91745.69645494515 and parameters: {'tau_C1': 58.452617689115776, 'tau_C2': 534.9477559004317, 'tau_C3': 444.6620156473909, 'tau_C4': 1726.0415744003326, 'tau_C5': 1853.1032629792874, 'tau_C6': 18298.224585081454}. Best is trial 450 with value: 84730.45864919.


[I 2026-04-25 16:34:35,126] Trial 470 finished with value: 85144.68797331351 and parameters: {'tau_C1': 50.117077201355464, 'tau_C2': 536.2148386102352, 'tau_C3': 475.7971047693399, 'tau_C4': 1885.4136362181264, 'tau_C5': 1803.1389407051688, 'tau_C6': 17839.516825229446}. Best is trial 450 with value: 84730.45864919.
[I 2026-04-25 16:34:35,131] Trial 471 finished with value: 115820.94245847836 and parameters: {'tau_C1': 53.60943951292978, 'tau_C2': 532.9321119967334, 'tau_C3': 28.6938425574864, 'tau_C4': 1928.3747887687578, 'tau_C5': 1756.1603811132982, 'tau_C6': 18775.138893440275}. Best is trial 450 with value: 84730.45864919.
[I 2026-04-25 16:34:35,137] Trial 472 finished with value: 95955.72628553123 and parameters: {'tau_C1': 63.88326442359134, 'tau_C2': 506.5265598064, 'tau_C3': 497.74057337667483, 'tau_C4': 1962.70231690447, 'tau_C5': 1882.6997376256302, 'tau_C6': 17830.353604125878}. Best is trial 450 with value: 84730.45864919.
[I 2026-04-25 16:34:35,142] Trial 473 finished wi

,number,value,datetime_start,datetime_complete,duration,params_tau_C1,params_tau_C2,params_tau_C3,params_tau_C4,params_tau_C5,params_tau_C6,user_attrs_annual_cost,user_attrs_availability,state
0,484,84507.284268,2026-04-25 16:34:35.197374,2026-04-25 16:34:35.202341,0 days 00:00:00.004967,50.040126,540.569694,473.874287,1836.217061,2462.544207,19920.895587,84507.284268,0.975166,COMPLETE
1,450,84730.458649,2026-04-25 16:34:35.011380,2026-04-25 16:34:35.016122,0 days 00:00:00.004742,50.109564,510.805937,498.468732,1627.853777,2019.173248,19635.027372,84730.458649,0.974987,COMPLETE
2,473,84744.439519,2026-04-25 16:34:35.137683,2026-04-25 16:34:35.142510,0 days 00:00:00.004827,50.144842,500.222833,471.398618,1803.599350,2077.893302,19983.046542,84744.439519,0.975011,COMPLETE
3,438,84751.544572,2026-04-25 16:34:34.946845,2026-04-25 16:34:34.952001,0 days 00:00:00.005156,50.022930,536.273283,496.718956,1498.080196,2094.015211,17580.780934,84751.544572,0.975002,COMPLETE
4,449,84775.064417,2026-04-25 16:34:35.006117,2026-04-25 16:34:35.010957,0 days 00:00:00.004840,50.151110,504.822218,499.068479,1484.725273,2010.232601,19944.373101,84775.064417,0.974974,COMPLETE


In [7]:
TOP_K = 5
real_results = []
for _, row in study_df.head(TOP_K).iterrows():
    tau_vector = {c: float(row[f'params_tau_{c}']) for c in COMPONENT_IDS}
    events = run_with_tau(
        tau_vector,
        printer_ids=TEST_PRINTERS,
        dates=DATES,
        components_cfg=components_cfg,
        couplings_cfg=couplings_cfg,
        cities_cfg=cities_cfg,
    )
    real_score = scalar_objective(events, components_cfg)
    real_results.append({
        'trial': int(row['number']),
        'surrogate_value': float(row['value']),
        'surrogate_annual_cost': float(row['user_attrs_annual_cost']),
        'real_value': float(real_score['value']),
        'real_annual_cost': float(real_score['annual_cost']),
        'real_availability': float(real_score['availability']),
        **{f'tau_{c}': tau_vector[c] for c in COMPONENT_IDS},
    })
real_df = pd.DataFrame(real_results).sort_values('real_value').reset_index(drop=True)
real_df

,trial,surrogate_value,surrogate_annual_cost,real_value,real_annual_cost,real_availability,tau_C1,tau_C2,tau_C3,tau_C4,tau_C5,tau_C6
0,484,84507.284268,84507.284268,1.050000e+10,4.507038e+06,0.0,50.040126,540.569694,473.874287,1836.217061,2462.544207,19920.895587
1,450,84730.458649,84730.458649,1.050000e+10,4.502872e+06,0.0,50.109564,510.805937,498.468732,1627.853777,2019.173248,19635.027372
2,473,84744.439519,84744.439519,1.050000e+10,4.507939e+06,0.0,50.144842,500.222833,471.398618,1803.599350,2077.893302,19983.046542
3,438,84751.544572,84751.544572,1.050000e+10,4.506208e+06,0.0,50.022930,536.273283,496.718956,1498.080196,2094.015211,17580.780934
4,449,84775.064417,84775.064417,1.050000e+10,4.504831e+06,0.0,50.151110,504.822218,499.068479,1484.725273,2010.232601,19944.373101


In [8]:
winner = real_df.iloc[0]
best_tau = {c: float(winner[f'tau_{c}']) for c in COMPONENT_IDS}
payload = {
    'tau_nom_h': best_tau,
    'validated_on': 'test printers (id 85..99)',
    'surrogate_annual_cost': float(winner['surrogate_annual_cost']),
    'real_annual_cost_eur_per_printer_year': float(winner['real_annual_cost']),
    'real_availability': float(winner['real_availability']),
}
with (RESULTS_DIR / 'best_tau_surrogate.yaml').open('w', encoding='utf-8') as handle:
    yaml.safe_dump(payload, handle, sort_keys=False)
print(yaml.safe_dump(payload, sort_keys=False))

tau_nom_h:
  C1: 50.0401256648205
  C2: 540.5696937855336
  C3: 473.87428737328895
  C4: 1836.2170607142916
  C5: 2462.544206639712
  C6: 19920.895587196745
validated_on: test printers (id 85..99)
surrogate_annual_cost: 84507.28426842293
real_annual_cost_eur_per_printer_year: 4507037.68683274
real_availability: 0.0

